# CNN: Optimización y Transfer Learning

En este notebook trabajamos con:

- Una CNN simple en MNIST.
- Una CNN más profunda y optimizada en CIFAR-10.
- Transfer learning con ResNet50 sobre CIFAR-10.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report



## Sección 1 — CNN simple en MNIST


In [2]:
# Cargar dataset MNIST
(x_train_m, y_train_m), (x_test_m, y_test_m) = keras.datasets.mnist.load_data()

# Normalizar a [0, 1]
x_train_m = x_train_m.astype('float32') / 255.0
x_test_m  = x_test_m.astype('float32') / 255.0

# Añadir canal (grayscale)
x_train_m = x_train_m[..., None]
x_test_m  = x_test_m[..., None]

input_shape_m = (28, 28, 1)
num_classes_m = 10

print('MNIST shapes:')
print('x_train:', x_train_m.shape, 'y_train:', y_train_m.shape)
print('x_test :', x_test_m.shape, 'y_test :', y_test_m.shape)



MNIST shapes:
x_train: (60000, 28, 28, 1) y_train: (60000,)
x_test : (10000, 28, 28, 1) y_test : (10000,)


In [3]:
def make_mnist_cnn():
    model = keras.Sequential([
        layers.Conv2D(32, 3, activation='relu', input_shape=input_shape_m),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes_m, activation='softmax'),
    ])
    return model

model_mnist = make_mnist_cnn()
model_mnist.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_mnist = model_mnist.fit(
    x_train_m, y_train_m,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1,
)



d:\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.9480 - loss: 0.1779 - val_accuracy: 0.9857 - val_loss: 0.0503
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9836 - loss: 0.0538 - val_accuracy: 0.9875 - val_loss: 0.0443
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.9890 - loss: 0.0374 - val_accuracy: 0.9858 - val_loss: 0.0473
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9912 - loss: 0.0285 - val_accuracy: 0.9900 - val_loss: 0.0320
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9924 - loss: 0.0229 - val_accuracy: 0.9902 - val_loss: 0.0355


In [4]:
# Evaluación avanzada en test (Precision, Recall, F1)
y_pred_probs_m = model_mnist.predict(x_test_m)
y_pred_m = np.argmax(y_pred_probs_m, axis=1)

print(classification_report(y_test_m, y_pred_m, digits=4))



313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
              precision    recall  f1-score   support

           0     0.9969    0.9929    0.9949       980
           1     0.9921    0.9974    0.9947      1135
           2     0.9971    0.9884    0.9927      1032
           3     0.9892    0.9960    0.9926      1010
           4     0.9909    0.9980    0.9944       982
           5     0.9910    0.9877    0.9893       892
           6     0.9968    0.9896    0.9932       958
           7     0.9913    0.9951    0.9932      1028
           8     0.9730    1.0000    0.9863       974
           9     0.9980    0.9703    0.9839      1009

    accuracy                         0.9916     10000
   macro avg     0.9916    0.9915    0.9915     10000
weighted avg     0.9917    0.9916    0.9916     10000



## Sección 2 — CNN más profunda y optimizada en CIFAR-10


In [5]:
from tensorflow.keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalizar
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

y_train = y_train.reshape(-1)
y_test  = y_test.reshape(-1)

from sklearn.model_selection import train_test_split

x_train_c, x_val_c, y_train_c, y_val_c = train_test_split(
    x_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

input_shape = (32, 32, 3)
num_classes = 10

print('CIFAR-10 shapes:')
print('x_train:', x_train_c.shape, 'y_train:', y_train_c.shape)
print('x_val  :', x_val_c.shape, 'y_val  :', y_val_c.shape)
print('x_test :', x_test.shape,  'y_test :', y_test.shape)



d:\.venv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


CIFAR-10 shapes:
x_train: (45000, 32, 32, 3) y_train: (45000,)
x_val  : (5000, 32, 32, 3) y_val  : (5000,)
x_test : (10000, 32, 32, 3) y_test : (10000,)


In [6]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])



In [7]:
def make_cifar_vgg_like():
    inputs = keras.Input(shape=input_shape)

    x = data_augmentation(inputs)

    # Bloque 1
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Bloque 2
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Bloque 3
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model

model_cifar_opt = make_cifar_vgg_like()
model_cifar_opt.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



In [8]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),
]

history_cifar_opt = model_cifar_opt.fit(
    x_train_c, y_train_c,
    epochs=50,
    batch_size=64,
    validation_data=(x_val_c, y_val_c),
    callbacks=callbacks,
    verbose=1,
)



Epoch 1/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 51s 69ms/step - accuracy: 0.3142 - loss: 1.8460 - val_accuracy: 0.4744 - val_loss: 1.4313 - learning_rate: 0.0010
Epoch 2/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 59s 84ms/step - accuracy: 0.4468 - loss: 1.5183 - val_accuracy: 0.4404 - val_loss: 1.6007 - learning_rate: 0.0010
Epoch 3/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 57s 82ms/step - accuracy: 0.5000 - loss: 1.3895 - val_accuracy: 0.5032 - val_loss: 1.4649 - learning_rate: 0.0010
Epoch 4/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 60s 85ms/step - accuracy: 0.5438 - loss: 1.2654 - val_accuracy: 0.5764 - val_loss: 1.1905 - learning_rate: 5.0000e-04
Epoch 5/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 57s 80ms/step - accuracy: 0.5696 - loss: 1.2104 - val_accuracy: 0.6216 - val_loss: 1.0596 - learning_rate: 5.0000e-04
Epoch 6/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 81s 79ms/step - accuracy: 0.5845 - loss: 1.1698 - val_accuracy: 0.6286 - val_loss: 1.0442 - learning_rate: 5.0000e-04
Epoch 7/50
704/704 ━━━━━━━━━━━━━━━━━━━━ 57s 81ms/step - accuracy

In [9]:
y_val_pred_probs = model_cifar_opt.predict(x_val_c)
y_val_pred = np.argmax(y_val_pred_probs, axis=1)

print('CNN VGG-like optimizada (validación):')
print(classification_report(y_val_c, y_val_pred, digits=4))

y_test_pred_probs = model_cifar_opt.predict(x_test)
y_test_pred = np.argmax(y_test_pred_probs, axis=1)

print('CNN VGG-like optimizada (test):')
print(classification_report(y_test, y_test_pred, digits=4))



157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step
CNN VGG-like optimizada (validación):
              precision    recall  f1-score   support

           0     0.7598    0.7340    0.7467       500
           1     0.8091    0.8900    0.8476       500
           2     0.6538    0.5400    0.5915       500
           3     0.5321    0.4640    0.4957       500
           4     0.6972    0.6080    0.6496       500
           5     0.6974    0.5300    0.6023       500
           6     0.5838    0.8780    0.7013       500
           7     0.7414    0.7800    0.7602       500
           8     0.8300    0.8400    0.8350       500
           9     0.8050    0.8340    0.8193       500

    accuracy                         0.7098      5000
   macro avg     0.7110    0.7098    0.7049      5000
weighted avg     0.7110    0.7098    0.7049      5000

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step
CNN VGG-like optimizada (test):
              precision    recall  f1-score   support

           0     0.7285    0.746

## Sección 3 — Transfer learning con ResNet50 en CIFAR-10


In [10]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

base_resnet = ResNet50(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)
base_resnet.trainable = False

inputs_tl = keras.Input(shape=input_shape)
x = layers.Resizing(224, 224)(inputs_tl)
x = preprocess_input(x)
x = base_resnet(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs_tl = layers.Dense(num_classes, activation='softmax')(x)

model_tl = keras.Model(inputs_tl, outputs_tl)
model_tl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



In [11]:
history_tl = model_tl.fit(
    x_train_c, y_train_c,
    epochs=10,
    batch_size=64,
    validation_data=(x_val_c, y_val_c),
    verbose=1,
)

y_val_pred_probs_tl = model_tl.predict(x_val_c)
y_val_pred_tl = np.argmax(y_val_pred_probs_tl, axis=1)

print('Transfer learning ResNet50 (validación, solo cabeza):')
print(classification_report(y_val_c, y_val_pred_tl, digits=4))



Epoch 1/10
315/704 ━━━━━━━━━━━━━━━━━━━━ 12:49 2s/step - accuracy: 0.1034 - loss: 2.3680

KeyboardInterrupt: 

In [ ]:
# Fine-tuning de las últimas capas de ResNet50
base_resnet.trainable = True
for layer in base_resnet.layers[:-20]:
    layer.trainable = False

model_tl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_tl_ft = model_tl.fit(
    x_train_c, y_train_c,
    epochs=10,
    batch_size=64,
    validation_data=(x_val_c, y_val_c),
    verbose=1,
)

y_val_pred_probs_tl_ft = model_tl.predict(x_val_c)
y_val_pred_tl_ft = np.argmax(y_val_pred_probs_tl_ft, axis=1)

print('Transfer learning ResNet50 (validación, fine-tuning):')
print(classification_report(y_val_c, y_val_pred_tl_ft, digits=4))

